# 🔄 Customer Churn Prediction — End-to-End ML Pipeline

> **Author:** Sourav Nayak | [LinkedIn](https://www.linkedin.com/in/sourav-nayak-756690306) | [GitHub](https://github.com/SOURAV033)
>
> **Tech Stack:** Python · Pandas · NumPy · scikit-learn · Matplotlib · Seaborn

---

## 📋 Project Overview

Customer churn is one of the most critical business problems for subscription-based companies. Losing a customer costs **5–25× more** than retaining one. This project builds a complete ML pipeline to:

1. **Analyze** customer behavior and identify churn drivers through EDA
2. **Predict** which customers are likely to churn using multiple ML models
3. **Compare** model performance across Accuracy, Precision, Recall, F1, and ROC-AUC
4. **Interpret** results and provide actionable business recommendations

### Dataset
- **7,043 telecom customers** with 20 features
- **Churn rate:** ~25.7% (realistic class imbalance)
- Features: demographics, services subscribed, billing info, contract type

### Models Trained
| Model | Strength |
|---|---|
| Logistic Regression | Interpretable baseline, handles imbalance |
| Random Forest | Handles non-linearity, feature importance |
| Gradient Boosting | High accuracy, sequential learning |
| SVM | Strong on small-medium datasets |

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle
warnings.filterwarnings('ignore')

from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.model_selection  import train_test_split, cross_val_score
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm              import SVC
from sklearn.metrics          import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Plot style
plt.rcParams.update({'figure.facecolor': '#F8F9FA', 'axes.facecolor': '#F8F9FA',
                     'font.family': 'DejaVu Sans', 'axes.spines.top': False,
                     'axes.spines.right': False})
CHURN_COLOR = '#E05B5B'
STAY_COLOR  = '#0A7B8C'

print('✅ All libraries loaded successfully!')
print(f'   pandas {pd.__version__} | numpy {np.__version__}')

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/telco_churn.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum()[df.isnull().sum()>0])
print('\n--- Target Distribution ---')
print(df['Churn'].value_counts())
print(f'\nChurn Rate: {(df["Churn"]=="Yes").mean()*100:.1f}%')

In [ ]:
df.describe(include='all').T.style.background_gradient(cmap='Blues', subset=['mean','std'])

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Churn rate donut
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Donut chart
counts = df['Churn'].value_counts()
axes[0].pie(counts, labels=None, colors=[STAY_COLOR, CHURN_COLOR],
            autopct='%1.1f%%', startangle=90, pctdistance=0.7,
            wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
axes[0].legend(['Retained','Churned'], loc='lower center', ncol=2, frameon=False)
axes[0].set_title('Overall Churn Rate', fontweight='bold')

# Tenure distribution
for label, color in [('No', STAY_COLOR), ('Yes', CHURN_COLOR)]:
    axes[1].hist(df[df['Churn']==label]['tenure'], bins=24, alpha=0.65,
                 color=color, label='Retained' if label=='No' else 'Churned', edgecolor='white')
axes[1].set_xlabel('Tenure (months)'); axes[1].set_ylabel('Count')
axes[1].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[1].legend()

# Monthly charges violin
sns.violinplot(data=df, x='Churn', y='MonthlyCharges',
               palette={'No': STAY_COLOR, 'Yes': CHURN_COLOR},
               inner='quartile', ax=axes[2], linewidth=1.2)
axes[2].set_xticklabels(['Retained','Churned'])
axes[2].set_title('Monthly Charges by Churn', fontweight='bold')
axes[2].set_xlabel('')

plt.suptitle('EDA — Core Churn Patterns', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/nb_eda1.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Business driver charts
df_tmp = df.copy()
df_tmp['Churn_Flag'] = (df_tmp['Churn']=='Yes').astype(int)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in zip(axes,
    ['Contract','InternetService','PaymentMethod'],
    ['Churn Rate by Contract','Churn Rate by Internet','Churn Rate by Payment']):
    d = df_tmp.groupby(col)['Churn_Flag'].mean().sort_values()*100
    bars = ax.bar(d.index, d.values, color=['#5B3FD4','#0A7B8C','#E05B5B','#F5A623'][:len(d)],
                  edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, d.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)
    ax.set_title(title, fontweight='bold'); ax.set_ylabel('Churn Rate (%)')
    ax.set_xticklabels(d.index, rotation=15, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('EDA — Business Driver Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/nb_eda2.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n💡 Key Insight: Month-to-month contracts have significantly higher churn!')

## 4. Data Cleaning & Feature Engineering

In [ ]:
df_clean = df.copy()

# Drop ID, fix types, encode target
df_clean.drop(columns=['customerID'], inplace=True)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0)
df_clean['Churn'] = (df_clean['Churn']=='Yes').astype(int)

# --- FEATURE ENGINEERING ---
# 1. Tenure lifecycle group
df_clean['tenure_group'] = pd.cut(df_clean['tenure'], bins=[-1,12,24,48,72],
    labels=['New (0-12m)','Growing (1-2yr)','Established (2-4yr)','Loyal (4yr+)'])

# 2. Average monthly spend efficiency
df_clean['avg_monthly_spend'] = df_clean['MonthlyCharges'] / (df_clean['tenure'] + 1)

# 3. Bundled services flag
df_clean['has_bundle'] = (
    (df_clean['PhoneService']=='Yes') &
    (df_clean['InternetService'].isin(['DSL','Fiber optic']))
).astype(int)

# 4. High-value customer flag (top 25% by monthly charges)
df_clean['high_value'] = (df_clean['MonthlyCharges'] > df_clean['MonthlyCharges'].quantile(0.75)).astype(int)

# 5. Contract risk score
df_clean['contract_risk'] = df_clean['Contract'].map({'Month-to-month':2,'One year':1,'Two year':0})

print(f'Features after engineering: {df_clean.shape[1]}')
print('\nNew feature sample:')
df_clean[['tenure','tenure_group','avg_monthly_spend','has_bundle','high_value','contract_risk']].head()

## 5. Preprocessing — Encode & Split

In [ ]:
y = df_clean['Churn']
X = df_clean.drop(columns=['Churn'])

# Label-encode categorical columns
le = LabelEncoder()
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))

# Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

# Scale numeric features
num_cols = ['tenure','MonthlyCharges','TotalCharges','avg_monthly_spend',
            'SeniorCitizen','has_bundle','high_value','contract_risk']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean()*100:.1f}%  |  Test churn rate: {y_test.mean()*100:.1f}%')

## 6. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression':  LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest':        RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced'),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    'SVM':                  SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1 Score':  round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
    })
    print(f'{name:<25}  F1={results[-1]["F1 Score"]:.4f}  AUC={results[-1]["ROC-AUC"]:.4f}')

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print('\n✅ All models trained!')

In [ ]:
# Display styled results table
results_df.style \
    .background_gradient(cmap='Greens', subset=['ROC-AUC','F1 Score']) \
    .highlight_max(subset=['ROC-AUC','F1 Score','Recall'], color='#c6efce') \
    .format(precision=4) \
    .set_caption('Model Performance Comparison')

## 7. Visualisations — Model Comparison

In [ ]:
# Bar chart comparison
palette = {'Logistic Regression':'#5B3FD4','Random Forest':'#0A7B8C',
           'Gradient Boosting':'#E05B5B','SVM':'#F5A623'}

metrics = ['Accuracy','Precision','Recall','F1 Score','ROC-AUC']
x = np.arange(len(metrics)); width = 0.18

fig, ax = plt.subplots(figsize=(12, 5))
for i, row in results_df.reset_index(drop=True).iterrows():
    vals   = [row[m] for m in metrics]
    offset = (i - len(results_df)/2 + 0.5) * width
    bars   = ax.bar(x+offset, vals, width, label=row['Model'],
                    color=palette[row['Model']], alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0.60, 1.02); ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', framealpha=0.9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../outputs/nb_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:,1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    ax.plot(fpr, tpr, color=palette[name], lw=2.2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'--', color='gray', lw=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('../outputs/nb_roc_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Best model: Feature importance (Random Forest)
rf_model = models['Random Forest']
fi = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
fi = fi.sort_values('Importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(9, 7))
colors  = ['#0A7B8C' if v > fi['Importance'].median() else '#B0C4DE' for v in fi['Importance']]
ax.barh(fi['Feature'], fi['Importance'], color=colors, edgecolor='white')
ax.set_xlabel('Feature Importance'); ax.set_title('Top 15 Features — Random Forest', fontweight='bold')
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('../outputs/nb_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTop 5 churn predictors:')
print(fi.tail(5)[['Feature','Importance']].to_string(index=False))

## 8. Business Recommendations

Based on model insights, here are actionable strategies to reduce churn:

| # | Insight | Recommendation |
|---|---------|----------------|
| 1 | **Short tenure = highest churn** | Onboarding rewards for first 12 months |
| 2 | **Month-to-month contracts** have 3× higher churn | Incentivise annual contract upgrades |
| 3 | **Fiber optic customers** churn more | Improve service reliability & add value |
| 4 | **Electronic check payers** churn more | Offer autopay discounts |
| 5 | **No tech support** customers churn more | Bundle tech support for high-value customers |

In [ ]:
# Churn risk by tenure group — actionable segment view
df['Churn_Flag'] = (df['Churn']=='Yes').astype(int)
df['tenure_group'] = pd.cut(df['tenure'], bins=[-1,12,24,48,72],
    labels=['New (0-12m)','Growing (1-2yr)','Established (2-4yr)','Loyal (4yr+)'])

seg = df.groupby('tenure_group')['Churn_Flag'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(seg.index, seg.values,
              color=[CHURN_COLOR,'#F5A623','#5B3FD4',STAY_COLOR],
              edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, seg.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{v:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_ylabel('Churn Rate (%)'); ax.set_xlabel('Customer Lifecycle Stage')
ax.set_title('Churn Rate by Customer Lifecycle — Priority Segments', fontweight='bold')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../outputs/nb_segment_churn.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Summary

| Metric | Best Model (Logistic Regression) |
|--------|----------------------------------|
| ROC-AUC | 0.6902 |
| F1 Score | 0.4819 |
| Recall | 0.6584 |
| Precision | 0.3800 |

**Key Takeaways:**
- The model successfully identifies ~66% of churners before they leave
- Tenure, contract type, and monthly charges are the strongest churn predictors
- Targeting new customers (0-12 months) with retention campaigns will yield the highest ROI

---
*Built by Sourav Nayak | B.Tech CSE, KIIT University | [github.com/SOURAV033](https://github.com/SOURAV033)*